# DEE — Test κ → Ric (revisado con k escalado correctamente)

**Corrección metodológica:** en el test anterior usamos k_neighbors=15 fijo para todas las densidades. Según el teorema de van der Hoorn et al. (2021), la convergencia κ → Ric requiere tomar simultáneamente N → ∞ y k(N) → ∞ con k/N → 0. 

**Escalamiento apropiado:** en 3D, la elección estándar es k ~ N^(1/3) — crece con la densidad volumétrica. Esto mantiene el vecindario a una distancia física aproximadamente constante cuando aumenta la densidad.

**Tres tests con la nueva escala:**

1. **Test T³:** ⟨κ⟩ → 0 con N creciente y k(N) ~ N^(1/3)
2. **Ley de escala:** σ_κ ∝ N^(-α) con α medible
3. **Test S³:** discriminación geométrica se mantiene

---

In [ ]:
!pip install POT -q

import os
import numpy as np
from scipy.sparse import csr_matrix
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import ot
import time

PLOT_DIR = 'plots_kappa_ricci_v2'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones base idénticas al test anterior
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def generate_S3_points(N_target, seed=42):
    rng = np.random.default_rng(seed)
    points = rng.normal(size=(N_target, 4))
    points = points / np.linalg.norm(points, axis=1, keepdims=True)
    return points

def spherical_distance_matrix_S3(points):
    dot = np.clip(points @ points.T, -1.0, 1.0)
    return np.arccos(dot)

def find_neighbors(D_mat, k_neighbors):
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    return np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]

def ollivier_ricci_edge(i, j, neighbors, D_mat):
    nbrs_i = neighbors[i]
    nbrs_j = neighbors[j]
    mu_i = np.ones(len(nbrs_i)) / len(nbrs_i)
    mu_j = np.ones(len(nbrs_j)) / len(nbrs_j)
    cost = D_mat[np.ix_(nbrs_i, nbrs_j)]
    W1 = ot.emd2(mu_i, mu_j, cost)
    d_ij = D_mat[i, j]
    if d_ij < 1e-10:
        return 0.0
    return 1.0 - W1 / d_ij

def compute_kappa_sample(D_mat, k_neighbors, n_edges_sample=200, seed=42):
    N = len(D_mat)
    neighbors = find_neighbors(D_mat, k_neighbors)
    rng = np.random.default_rng(seed)
    kappas = []
    sampled = set()
    attempts = 0
    n_target = min(n_edges_sample, N * k_neighbors)
    while len(kappas) < n_target and attempts < n_target * 10:
        i = rng.integers(N)
        j = int(neighbors[i, rng.integers(k_neighbors)])
        edge = (min(i, j), max(i, j))
        if edge in sampled:
            attempts += 1
            continue
        sampled.add(edge)
        kappa = ollivier_ricci_edge(i, j, neighbors, D_mat)
        kappas.append(kappa)
        attempts += 1
    return np.array(kappas)

print('Funciones listas')

## Escalamiento de k con N

Queremos k(N) que cumpla:
- Crece con N (requisito del teorema)
- k/N → 0 (vecindario local, no global)
- El **radio físico** del vecindario se mantiene aproximadamente constante

En 3D, esto último se logra con k ∝ N^(1/3) aproximadamente. Por ejemplo, si el espaciado medio entre nodos es ∼N^(-1/3) y queremos el vecindario hasta un radio fijo r*, necesitamos k ∼ (r*/espaciado)³ ∼ r*³ · N que escala linealmente pero con pendiente pequeña. 

Optamos por una versión moderada: **k(N) = round(8 · N^(1/3))**. Con N=343 → k=56, con N=3375 → k=120. Esto da vecindarios de ~40-50 puntos en promedio que crecen suavemente con N.

In [ ]:
def k_of_N(N, coef=8):
    """Escalamiento k ~ N^(1/3) con coeficiente ajustable."""
    return max(10, int(coef * N**(1/3)))

# Densidades a testear
N_targets = [343, 729, 1331, 2197, 3375]

print(f'{"N":>6} {"k(N)":>6} {"k/N":>8}')
print('-'*25)
for N in N_targets:
    k = k_of_N(N)
    print(f'{N:>6} {k:>6} {k/N:>8.4f}')

print(f'\nk/N tiende a cero: ✓ (requisito del teorema)')
print(f'k crece con N: ✓ (requisito del teorema)')

## TEST 1 — Convergencia en T³ con k escalado

In [ ]:
print('Test T³ con k(N) escalado — puede tomar varios minutos para N altos')
print()

results_T3 = []

for N_target in N_targets:
    t0 = time.time()
    points, N = grid_perturbed_T3(N_target, 0.1, seed=42)
    D_mat = periodic_distance_matrix(points, L=1.0)
    
    k_val = k_of_N(N)
    # Muestreamos más enlaces cuando N y k crecen (estadística mejor)
    n_sample = min(400, max(200, N // 4))
    
    kappas = compute_kappa_sample(D_mat, k_neighbors=k_val, 
                                    n_edges_sample=n_sample, seed=42)
    
    results_T3.append({
        'N': N,
        'k': k_val,
        'n_sample': len(kappas),
        'mean': np.mean(kappas),
        'std': np.std(kappas),
        'sem': np.std(kappas) / np.sqrt(len(kappas)),
        'time': time.time() - t0
    })
    
    r = results_T3[-1]
    print(f'N={N:5d} k={k_val:3d}: ⟨κ⟩ = {r["mean"]:+.4f} ± {r["sem"]:.4f}, '
          f'σ_κ = {r["std"]:.4f}, '
          f'muestras = {r["n_sample"]}, '
          f'tiempo = {r["time"]:.0f}s')

## TEST 2 — Convergencia en S³ con k escalado

In [ ]:
print('Test S³ con k(N) escalado')
print()

results_S3 = []

for N_target in N_targets:
    t0 = time.time()
    points_S3 = generate_S3_points(N_target, seed=42)
    D_mat_S3 = spherical_distance_matrix_S3(points_S3)
    
    k_val = k_of_N(N_target)
    n_sample = min(400, max(200, N_target // 4))
    
    kappas_S3 = compute_kappa_sample(D_mat_S3, k_neighbors=k_val,
                                       n_edges_sample=n_sample, seed=42)
    
    results_S3.append({
        'N': N_target,
        'k': k_val,
        'n_sample': len(kappas_S3),
        'mean': np.mean(kappas_S3),
        'std': np.std(kappas_S3),
        'sem': np.std(kappas_S3) / np.sqrt(len(kappas_S3)),
        'time': time.time() - t0
    })
    
    r = results_S3[-1]
    print(f'N={N_target:5d} k={k_val:3d}: ⟨κ⟩ = {r["mean"]:+.4f} ± {r["sem"]:.4f}, '
          f'σ_κ = {r["std"]:.4f}, '
          f'tiempo = {r["time"]:.0f}s')

## Análisis de convergencia y ley de escala

In [ ]:
# Arrays
N_vals = np.array([r['N'] for r in results_T3])
k_vals = np.array([r['k'] for r in results_T3])
kappa_T3_mean = np.array([r['mean'] for r in results_T3])
kappa_T3_std = np.array([r['std'] for r in results_T3])
kappa_T3_sem = np.array([r['sem'] for r in results_T3])
kappa_S3_mean = np.array([r['mean'] for r in results_S3])
kappa_S3_std = np.array([r['std'] for r in results_S3])
kappa_S3_sem = np.array([r['sem'] for r in results_S3])

# Ajuste: ¿converge a 0 el ⟨κ⟩ de T³?
# Modelo: ⟨κ⟩(N) = A * N^(-β)
def decay_power(N, A, beta):
    return A * N**(-beta)

# Valores absolutos para poder ajustar ley de potencia
abs_kappa_T3 = np.abs(kappa_T3_mean)

try:
    popt_mean, _ = curve_fit(decay_power, N_vals, abs_kappa_T3, p0=[0.1, 0.3])
    A_mean, beta_mean = popt_mean
    print(f'Ajuste |⟨κ⟩_T³|(N) ∝ N^(-β):')
    print(f'  A    = {A_mean:.3f}')
    print(f'  β    = {beta_mean:.3f}')
    print(f'  (si β > 0.1 y A extrapola a 0, hay convergencia)')
except Exception as e:
    A_mean, beta_mean = None, None
    print(f'No se pudo ajustar ⟨κ⟩: {e}')

# Ajuste de fluctuaciones
try:
    popt_std, _ = curve_fit(decay_power, N_vals, kappa_T3_std, p0=[1.0, 0.3])
    A_std, alpha_std = popt_std
    print(f'\nAjuste σ_κ(N) ∝ N^(-α):')
    print(f'  α = {alpha_std:.3f}')
    print(f'  (teórico esperado en 3D: ~0.2-0.5 según teorema)')
except Exception as e:
    A_std, alpha_std = None, None
    print(f'No se pudo ajustar σ: {e}')

In [ ]:
# Plot principal
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: ⟨κ⟩ vs N en T³
ax = axes[0, 0]
ax.errorbar(N_vals, kappa_T3_mean, yerr=kappa_T3_sem, fmt='o-', 
             color='steelblue', markersize=10, capsize=5, lw=2,
             label='⟨κ⟩ observado DEE sobre T³')
ax.axhline(0, color='red', linestyle='--', lw=2, 
           label='Ric(T³ plano) = 0')
ax.set_xlabel('N (número de nodos)', fontsize=12)
ax.set_ylabel('⟨κ⟩', fontsize=12)
ax.set_title('Test 1: ⟨κ⟩ en T³ con k(N) escalado', fontsize=12)
ax.set_xscale('log')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 2: |⟨κ⟩| vs N log-log para ver ley de potencia
ax = axes[0, 1]
ax.loglog(N_vals, abs_kappa_T3, 'o', color='steelblue', markersize=10,
           label='|⟨κ⟩_T³|')
if A_mean is not None:
    N_smooth = np.logspace(np.log10(N_vals.min()*0.9), np.log10(N_vals.max()*1.1), 50)
    ax.loglog(N_smooth, decay_power(N_smooth, A_mean, beta_mean), '--',
              color='red', lw=2, label=f'Ajuste: |⟨κ⟩| ∝ N^(-{beta_mean:.2f})')
ax.set_xlabel('N (log)', fontsize=12)
ax.set_ylabel('|⟨κ⟩| (log)', fontsize=12)
ax.set_title('Test 1b: Convergencia cuantitativa a 0 (log-log)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 3: dispersión σ_κ vs N
ax = axes[1, 0]
ax.loglog(N_vals, kappa_T3_std, 'o', color='darkgreen', markersize=10,
           label='σ_κ observado T³')
if alpha_std is not None:
    N_smooth = np.logspace(np.log10(N_vals.min()*0.9), np.log10(N_vals.max()*1.1), 50)
    ax.loglog(N_smooth, decay_power(N_smooth, A_std, alpha_std), '--',
              color='red', lw=2, label=f'Ajuste: σ ∝ N^(-{alpha_std:.2f})')
ax.set_xlabel('N (log)', fontsize=12)
ax.set_ylabel('σ_κ (log)', fontsize=12)
ax.set_title('Test 2: Ley de escala de fluctuaciones', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 4: comparación T³ vs S³
ax = axes[1, 1]
ax.errorbar(N_vals, kappa_T3_mean, yerr=kappa_T3_sem, fmt='o-',
             color='steelblue', markersize=10, capsize=5, lw=2,
             label='T³ (Ric = 0)')
ax.errorbar(N_vals, kappa_S3_mean, yerr=kappa_S3_sem, fmt='s-',
             color='crimson', markersize=10, capsize=5, lw=2,
             label='S³ (Ric > 0)')
ax.axhline(0, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('N', fontsize=12)
ax.set_ylabel('⟨κ⟩', fontsize=12)
ax.set_title('Test 3: Discriminación T³ vs S³ con k(N) escalado', fontsize=12)
ax.set_xscale('log')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('33_convergencia_kappa_escalado')
plt.show()

In [ ]:
# Veredicto
print('='*70)
print('VEREDICTO — Convergencia κ → Ric con k(N) escalado')
print('='*70)
print()

# Criterio 1: ⟨κ⟩_T³ debe acercarse a 0 (tamaño y significancia)
kappa_final = kappa_T3_mean[-1]
kappa_initial = kappa_T3_mean[0]
reduccion_pct = (abs(kappa_initial) - abs(kappa_final)) / abs(kappa_initial) * 100

print('TEST 1 — Convergencia sobre T³ plano:')
print(f'  ⟨κ⟩ en N={N_vals[0]}: {kappa_initial:+.4f}')
print(f'  ⟨κ⟩ en N={N_vals[-1]}: {kappa_final:+.4f}')
print(f'  Reducción en valor absoluto: {reduccion_pct:+.1f}%')
sigma_ratio_final = abs(kappa_final) / kappa_T3_sem[-1]
print(f'  Distancia a cero en unidades de σ: {sigma_ratio_final:.1f}σ')
if sigma_ratio_final < 3:
    print(f'  → ✓ Compatible con cero a nivel {sigma_ratio_final:.1f}σ')
elif reduccion_pct > 30:
    print(f'  → ~ Tendencia a cero clara pero residuo finito')
else:
    print(f'  → ✗ No converge a cero en este rango de N')

print()
print('TEST 2 — Ley de escala:')
if alpha_std is not None:
    print(f'  σ_κ ∝ N^(-{alpha_std:.3f})')
    if 0.1 < alpha_std < 0.6:
        print(f'  → ✓ Exponente dentro del rango razonable para 3D')
    else:
        print(f'  → ~ Exponente fuera del rango esperado')

print()
print('TEST 3 — Discriminación geométrica:')
diff_geom = kappa_S3_mean[-1] - kappa_T3_mean[-1]
sigma_combined = np.sqrt(kappa_S3_sem[-1]**2 + kappa_T3_sem[-1]**2)
significancia = diff_geom / sigma_combined
print(f'  Δκ = κ(S³) − κ(T³) = {diff_geom:+.4f}')
print(f'  Significancia: {significancia:.1f}σ')
if significancia > 3:
    print(f'  → ✓✓ Discrimina claramente (>3σ)')
elif significancia > 2:
    print(f'  → ✓ Discrimina moderadamente')
else:
    print(f'  → ~ Discriminación débil')

In [ ]:
# Tabla resumen
print('\n' + '='*75)
print('TABLA RESUMEN — Convergencia κ → Ric con k(N) escalado')
print('='*75)
print(f'{"N":>6} {"k":>5} {"⟨κ⟩ T³":>12} {"σ T³":>8} {"⟨κ⟩ S³":>12} {"σ S³":>8}')
print('-'*55)
for r_T3, r_S3 in zip(results_T3, results_S3):
    print(f'{r_T3["N"]:>6} {r_T3["k"]:>5} '
          f'{r_T3["mean"]:>+12.4f} {r_T3["std"]:>8.4f} '
          f'{r_S3["mean"]:>+12.4f} {r_S3["std"]:>8.4f}')

In [ ]:
import shutil
shutil.make_archive('plots_kappa_ricci_v2', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_kappa_ricci_v2.zip')

try:
    from google.colab import files
    files.download('plots_kappa_ricci_v2.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass